# RTD-Net — AID Scene Classification
**Paper:** *Real-Time Object Detection Network in UAV-Vision Based on CNN and Transformer*  
Ye et al., IEEE TIM Vol. 72, 2023

**Setup:** Kaggle 2×T4 GPU · PyTorch · AID dataset · 50/50 train/val split  
**Hyperparams:** from paper image (SGDM, lr=0.01, momentum=0.937, wd=0.0005, 300 epochs, bs=64)

In [1]:
# ── 0. Sanity-check environment ────────────────────────────────────────────
import subprocess, sys
print(subprocess.check_output('nvidia-smi', shell=True).decode())
print('Python', sys.version)

Fri Apr 17 11:29:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import os, json, math, random, time, pathlib
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder

from PIL import Image
import matplotlib.pyplot as plt

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
# ── 2. Config ──────────────────────────────────────────────────────────────
CFG = {
    # Data
    'data_dir':    '/kaggle/input/aid-dataset',   # adjust if your dataset path differs
    'img_size':    224,
    'train_ratio': 0.5,          # 50 / 50 split  (standard AID benchmark protocol)
    'num_classes': 30,           # AID has 30 scene categories

    # Model
    'base_ch':   32,
    'num_heads':  4,
    'C':         16,             # LEM branches
    'dropout':    0.3,

    # Training  (from paper image)
    'epochs':       300,
    'batch_size':    128,         # paper: bs=64  (per GPU; DataParallel will aggregate)
    'lr':           0.01,
    'momentum':     0.937,
    'weight_decay': 0.0005,
    'lr_drop_every': 100,        # paper: lr ×0.1 every 100 epochs
    'lr_gamma':      0.1,
    'label_smoothing': 0.1,
    'patience':      50,         # early stopping

    # Output
    'save_dir': '/kaggle/working/runs',
    'seed':      42,
}

# ── Reproducibility ─────────────────────────────────────────────────────────
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()   # should be 2 on Kaggle 2×T4
print(f'Device: {DEVICE}  |  GPUs: {N_GPUS}')

save_dir = pathlib.Path(CFG['save_dir'])
save_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

Device: cuda  |  GPUs: 2
{
  "data_dir": "/kaggle/input/aid-dataset",
  "img_size": "224",
  "train_ratio": "0.5",
  "num_classes": "30",
  "base_ch": "32",
  "num_heads": "4",
  "C": "16",
  "dropout": "0.3",
  "epochs": "300",
  "batch_size": "128",
  "lr": "0.01",
  "momentum": "0.937",
  "weight_decay": "0.0005",
  "lr_drop_every": "100",
  "lr_gamma": "0.1",
  "label_smoothing": "0.1",
  "patience": "50",
  "save_dir": "/kaggle/working/runs",
  "seed": "42"
}


In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

DATA_DIR = "/kaggle/input/datasets/jiayuanchengala/aid-scene-classification-datasets/AID"

train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

full_ds = datasets.ImageFolder(DATA_DIR, transform=train_tfms)

train_size = int(0.5 * len(full_ds))
val_size = len(full_ds) - train_size

train_ds, val_ds = random_split(full_ds, [train_size, val_size])

In [12]:
BS = 128
NW = 4
NC = 30

train_loader = DataLoader(
    train_ds,
    batch_size=BS,
    shuffle=True,
    num_workers=NW,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BS * 2,
    shuffle=False,
    num_workers=NW,
    pin_memory=True
)

In [15]:
# ── 4. Model — RTD-Net modules (FIXED) ────────────────────────────────────

# ---- Helper ----------------------------------------------------------------
class ConvBNSiLU(nn.Module):
    """Standard Conv → BN → SiLU block."""
    def __init__(self, in_ch, out_ch, kernel=1, stride=1, padding=None, groups=1):
        super().__init__()
        if padding is None:
            padding = kernel // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, stride, padding,
                              groups=groups, bias=False)
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


# ---- LEM (Section III-B, Eq. 1) -------------------------------------------
class LEM(nn.Module):
    """
    Lightweight Feature Extraction Module.
    Homogeneous multi-branch: input → 1×1 conv → C branches (1×1 + 3×3) → cat → 1×1 → BN
    with residual skip.
    BUG FIXED: branches now receive a *shared* stem output (feat), not stacked inputs.
    """
    def __init__(self, in_ch, out_ch=None, C=16):
        super().__init__()
        out_ch = out_ch or in_ch
        mid_ch = max(in_ch // 2, 1)    # c/2
        br_ch  = max(in_ch // 32, 1)   # c/32 per branch
        self.C = C

        self.conv1 = ConvBNSiLU(in_ch, mid_ch, 1)

        # C homogeneous branches, each: 1×1 (mid_ch→br_ch) + 3×3 (br_ch→br_ch)
        self.branches = nn.ModuleList([
            nn.Sequential(
                ConvBNSiLU(mid_ch, br_ch, 1),
                ConvBNSiLU(br_ch,  br_ch, 3),
            )
            for _ in range(C)
        ])

        branch_total = br_ch * C
        self.conv2 = nn.Conv2d(branch_total, out_ch, 1, bias=False)
        self.bn    = nn.BatchNorm2d(out_ch)

        self.skip = nn.Identity() if in_ch == out_ch else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        feat = self.conv1(x)                               # shared stem
        outs = torch.cat([b(feat) for b in self.branches], dim=1)  # (B, br_ch*C, H, W)
        out  = self.bn(self.conv2(outs))
        return F.silu(out + self.skip(x))                  # residual


# ---- CMHSA (Section III-C, Eq. 2-3) — FIXED --------------------------------
class CMHSA(nn.Module):
    """
    Convolutional Multi-Head Self-Attention.

    BUG FIXED (original code had 2 bugs):
      1. Duplicate / wrong einsum: first einsum was wrong and immediately
         overwritten by the correct one.  Removed the dead first einsum.
      2. head_conv applied to (B, heads, T, T) where T can be large;
         this is correct architecturally but the reshape was redundant.
         Kept clean.
    """
    def __init__(self, dim, num_heads=4):
        super().__init__()
        assert dim % num_heads == 0, 'dim must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        # 1×1 conv projections for Q, K, V
        self.conv_q = nn.Conv2d(dim, dim, 1, bias=False)
        self.conv_k = nn.Conv2d(dim, dim, 1, bias=False)
        self.conv_v = nn.Conv2d(dim, dim, 1, bias=False)

        # Conv across heads (Eq. 3)
        self.head_conv = nn.Conv2d(num_heads, num_heads, 1, bias=False)

        # Instance norm after softmax (Eq. 3)
        self.inst_norm = nn.InstanceNorm2d(num_heads, affine=True)

        # Output projection
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        T = H * W

        # Convolutional Q/K/V projection + flatten  (Eq. 2)
        q = self.conv_q(x).flatten(2)           # (B, C, T)
        k = self.conv_k(x).flatten(2)           # (B, C, T)
        v = self.conv_v(x).flatten(2)           # (B, C, T)

        # Reshape into heads: (B, heads, head_dim, T)
        q = q.view(B, self.num_heads, self.head_dim, T)
        k = k.view(B, self.num_heads, self.head_dim, T)
        v = v.view(B, self.num_heads, self.head_dim, T)

        # ── FIX: single correct einsum for QK^T ──────────────────────────────
        # q: (B, h, d, Tq)  k: (B, h, d, Tk)  →  attn: (B, h, Tq, Tk)
        attn = torch.einsum('bhdq, bhdk -> bhqk', q, k) * self.scale  # (B, h, T, T)

        # Conv across heads on attention logits (Eq. 3)
        # Reshape to (B, h, T, T) — already that shape; head_conv treats h as channels
        attn = self.head_conv(attn)             # (B, h, T, T)

        # Softmax then instance norm  (Eq. 3)
        attn = F.softmax(attn, dim=-1)          # (B, h, T, T)
        attn = self.inst_norm(attn)             # (B, h, T, T)  — IN treats h as channels

        # Weighted sum of V
        # v: (B, h, d, T) → need (B, h, T, d)
        v_t = v.permute(0, 1, 3, 2)            # (B, h, T, d)
        out = torch.einsum('bhqk, bhkd -> bhqd', attn, v_t)  # (B, h, T, d)

        # Merge heads → (B, T, C)
        out = out.contiguous().view(B, T, C)

        # Linear projection + reshape to spatial
        out = self.proj(out)                    # (B, T, C)
        out = out.permute(0, 2, 1).view(B, C, H, W)
        return out


# ---- ECTB (Section III-C, Fig. 3c) -----------------------------------------
class ECTB(nn.Module):
    """
    Efficient Convolutional Transformer Block.
    input → 1×1Conv (c→c/2) → CMHSA → 1×1Conv (c/2→c) → BN + residual
    """
    def __init__(self, in_ch, out_ch=None, num_heads=4):
        super().__init__()
        out_ch = out_ch or in_ch
        mid_ch = max(in_ch // 2, 1)

        self.conv1  = ConvBNSiLU(in_ch, mid_ch, 1)
        self.cmhsa  = CMHSA(mid_ch, num_heads=num_heads)
        self.conv2  = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.bn     = nn.BatchNorm2d(out_ch)

        self.skip = nn.Identity() if in_ch == out_ch else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        feat = self.conv1(x)
        feat = self.cmhsa(feat)
        feat = self.bn(self.conv2(feat))
        return F.silu(feat + self.skip(x))


# ---- NAM (Section III-E, Eq. 7-9) ------------------------------------------
class NAMChannelAttention(nn.Module):
    """Channel attention via BN scale factor γ  (Eq. 8)."""
    def __init__(self, channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels)

    def forward(self, x):
        normed = self.bn(x)
        gamma  = self.bn.weight.abs()               # (C,)
        w      = gamma / (gamma.sum() + 1e-8)       # normalised importance
        w      = w.view(1, -1, 1, 1)
        return x * torch.sigmoid(w * normed)


class NAMSpatialAttention(nn.Module):
    """Spatial attention via pixel-level InstanceNorm  (Eq. 9)."""
    def __init__(self, channels):
        super().__init__()
        self.bn = nn.InstanceNorm2d(channels, affine=True)

    def forward(self, x):
        normed = self.bn(x)
        lam    = self.bn.weight.abs()               # (C,)
        w      = lam / (lam.sum() + 1e-8)
        w      = w.view(1, -1, 1, 1)
        return x * torch.sigmoid(w * normed)


class NAM(nn.Module):
    """Full NAM: channel → spatial attention."""
    def __init__(self, channels):
        super().__init__()
        self.channel = NAMChannelAttention(channels)
        self.spatial = NAMSpatialAttention(channels)

    def forward(self, x):
        return self.spatial(self.channel(x))


# ---- APH (Section III-E) ----------------------------------------------------
class APH(nn.Module):
    """Attention Prediction Head: NAM → 1×1Conv."""
    def __init__(self, in_ch, out_ch=None):
        super().__init__()
        out_ch = out_ch or in_ch
        self.nam  = NAM(in_ch)
        self.conv = ConvBNSiLU(in_ch, out_ch, 1)

    def forward(self, x):
        return self.conv(self.nam(x))


# ── 4b. Full classification backbone ─────────────────────────────────────────
class RTDNetClassifier(nn.Module):
    """
    RTD-Net adapted for scene classification.
    Backbone:  stem → LEM stack → ECTB block → APH head → Global Avg Pool → FC
    """
    def __init__(self, num_classes=30, base_ch=32, C=16, num_heads=4, dropout=0.3):
        super().__init__()
        bc = base_ch   # 32

        # Stem: 224 → 112
        self.stem = nn.Sequential(
            ConvBNSiLU(3,    bc,   3, stride=2),   # 112×112
            ConvBNSiLU(bc,   bc,   3, stride=1),
            ConvBNSiLU(bc,   bc*2, 3, stride=1),   # 112×112, 64ch
        )

        # Stage 1: LEM × 3  (112→56)
        self.stage1 = nn.Sequential(
            nn.MaxPool2d(2, 2),                     # 56×56
            LEM(bc*2,  bc*2,  C=C),
            LEM(bc*2,  bc*2,  C=C),
            LEM(bc*2,  bc*4,  C=C),                 # 56×56, 128ch
        )

        # Stage 2: LEM × 3  (56→28)
        self.stage2 = nn.Sequential(
            nn.MaxPool2d(2, 2),                     # 28×28
            LEM(bc*4,  bc*4,  C=C),
            LEM(bc*4,  bc*4,  C=C),
            LEM(bc*4,  bc*8,  C=C),                 # 28×28, 256ch
        )

        # Stage 3: ECTB  (28→14)
        self.stage3 = nn.Sequential(
            nn.MaxPool2d(2, 2),                     # 14×14
            ECTB(bc*8, bc*8,  num_heads=num_heads), # 14×14, 256ch
            ECTB(bc*8, bc*16, num_heads=num_heads), # 14×14, 512ch
        )

        # APH + classifier
        self.aph  = APH(bc*16, bc*16)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(bc*16, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.aph(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


# ── 4c. Quick shape sanity-check ──────────────────────────────────────────────
with torch.no_grad():
    _dummy = torch.randn(2, 3, 224, 224)
    _m = RTDNetClassifier(num_classes=30)
    _out = _m(_dummy)
    print(f'Model output shape: {_out.shape}  (expected [2, 30])')
    assert _out.shape == (2, 30)
    params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f'Trainable parameters: {params/1e6:.2f}M')
del _dummy, _m, _out

Model output shape: torch.Size([2, 30])  (expected [2, 30])
Trainable parameters: 0.88M


In [16]:
# ── 5. Build model + wrap with DataParallel for 2×T4 ─────────────────────
model = RTDNetClassifier(
    num_classes=NC,
    base_ch=CFG['base_ch'],
    C=CFG['C'],
    num_heads=CFG['num_heads'],
    dropout=CFG['dropout'],
)

# DataParallel automatically splits batches across all available GPUs
if N_GPUS > 1:
    print(f'Wrapping model with DataParallel across {N_GPUS} GPUs')
    model = nn.DataParallel(model)

model = model.to(DEVICE)
print(model)

Wrapping model with DataParallel across 2 GPUs
DataParallel(
  (module): RTDNetClassifier(
    (stem): Sequential(
      (0): ConvBNSiLU(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): ConvBNSiLU(
        (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): ConvBNSiLU(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    )
    (stage1): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1)

In [17]:
# ── 6. Optimiser / Scheduler / Loss  (paper hyperparams) ─────────────────
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=CFG['lr'],
    momentum=CFG['momentum'],
    weight_decay=CFG['weight_decay'],
    nesterov=True,          # Nesterov momentum — common improvement to SGDM
)

# lr × 0.1 every 100 epochs  (as stated in paper image)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[100, 200],
    gamma=CFG['lr_gamma'],
)

# Label smoothing cross-entropy
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

# Mixed precision for T4 (fp16 gives ~2× speed on Tensor Cores)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
print('Optimizer:', optimizer)
print('Scheduler milestones: [100, 200], gamma=0.1')

Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.01
    lr: 0.01
    maximize: False
    momentum: 0.937
    nesterov: True
    weight_decay: 0.0005
)
Scheduler milestones: [100, 200], gamma=0.1


In [21]:
# ── 7. Training & validation functions ───────────────────────────────────
def accuracy(output, target, topk=(1, 5)):
    """Returns top-k accuracy for each k."""
    with torch.no_grad():
        maxk = max(topk)
        batch = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred    = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        return [correct[:k].reshape(-1).float().sum(0) / batch * 100 for k in topk]


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, use_mixup=True):
    model.train()
    total_loss = 0.0
    total_top1 = 0.0
    n_batches  = len(loader)

    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)        
        labels = labels.to(DEVICE, non_blocking=True)

        # Mixup augmentation
        if use_mixup:
            imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.2)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):            
            logits = model(imgs)
            if use_mixup:
                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        # Gradient clipping (helps stability)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            top1 = accuracy(logits, labels if not use_mixup else labels_a)[0]

        total_loss += loss.item()
        total_top1 += top1.item()

        if (i + 1) % 20 == 0 or i == n_batches - 1:
            print(f'  Epoch {epoch:3d} [{i+1:3d}/{n_batches}]  '
                  f'loss={total_loss/(i+1):.4f}  top1={total_top1/(i+1):.2f}%  '
                  f'lr={optimizer.param_groups[0]["lr"]:.5f}')

    return total_loss / n_batches, total_top1 / n_batches


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    n_batches  = len(loader)

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)        
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):            
            logits = model(imgs)
            loss   = criterion(logits, labels)

        top1, top5 = accuracy(logits, labels, topk=(1, 5))
        total_loss += loss.item()
        total_top1 += top1.item()
        total_top5 += top5.item()

    return total_loss / n_batches, total_top1 / n_batches, total_top5 / n_batches

In [ ]:
# ── CELL: Mixup helpers + Checkpoint Resume ────────────────────────────────

# ── 1. Mixup functions (were missing → caused NameError) ──────────────────
def mixup_data(x, y, alpha=0.2):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss: weighted combination of two cross-entropy losses."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ── 2. DataLoaders (define here if not already defined above) ──────────────
NC = len(full_ds.classes)   # 30 for AID

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


# ── 3. Resume from checkpoint if it exists ────────────────────────────────
start_epoch    = 1
best_top1      = 0.0
best_epoch     = 0
patience_count = 0

if ckpt_path.exists():
    print(f"► Resuming from checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    
    # Load model weights (handle DataParallel wrapping)
    state = ckpt['model']
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(state, strict=False)
    else:
        model.load_state_dict(state, strict=False)
    
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    
    start_epoch    = ckpt['epoch'] + 1   # resume AFTER last saved epoch
    best_top1      = ckpt['val_top1']
    best_epoch     = ckpt['epoch']
    
    print(f"  Resumed at epoch {start_epoch}  |  best val top-1 so far: {best_top1:.2f}%")
else:
    print("No checkpoint found — training from scratch.")


# ── 4. Training loop (use start_epoch instead of 1) ───────────────────────
history = {
    'train_loss': [], 'train_top1': [],
    'val_loss':   [], 'val_top1':   [], 'val_top5': [],
}
t0 = time.time()

for epoch in range(start_epoch, CFG['epochs'] + 1):  # ← start_epoch, not 1
    tr_loss, tr_top1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch,
        use_mixup=(epoch > 5),
    )
    val_loss, val_top1, val_top5 = validate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_top1'].append(tr_top1)
    history['val_loss'].append(val_loss)
    history['val_top1'].append(val_top1)
    history['val_top5'].append(val_top5)

    elapsed = (time.time() - t0) / 60
    print(f'Epoch {epoch:3d}/{CFG["epochs"]}  '
          f'tr_loss={tr_loss:.4f} tr_top1={tr_top1:.2f}%  '
          f'val_loss={val_loss:.4f} val_top1={val_top1:.2f}%  '
          f'val_top5={val_top5:.2f}%  ({elapsed:.1f} min)')

    if val_top1 > best_top1:
        best_top1      = val_top1
        best_epoch     = epoch
        patience_count = 0
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'val_top1':  val_top1,
            'cfg':       CFG,
        }, ckpt_path)
        print(f'  ✓ New best: {val_top1:.2f}%  (saved)')
    else:
        patience_count += 1

    if patience_count >= CFG['patience']:
        print(f'Early stopping @ epoch {epoch}  (best: epoch {best_epoch} → {best_top1:.2f}%)')
        break

print(f'\nDone.  Best val Top-1: {best_top1:.2f}%  @ epoch {best_epoch}')

► Resuming from checkpoint: /kaggle/working/runs/best_model.pth
  Resumed at epoch 6  |  best val top-1 so far: 46.57%
  Epoch   6 [ 20/39]  loss=2.1485  top1=25.74%  lr=0.01000
  Epoch   6 [ 39/39]  loss=2.0576  top1=26.92%  lr=0.01000
Epoch   6/300  tr_loss=2.0576 tr_top1=26.92%  val_loss=2.2298 val_top1=41.54%  val_top5=79.30%  (1.1 min)
  Epoch   7 [ 20/39]  loss=2.0809  top1=35.04%  lr=0.01000
  Epoch   7 [ 39/39]  loss=2.0267  top1=33.99%  lr=0.01000
Epoch   7/300  tr_loss=2.0267 tr_top1=33.99%  val_loss=1.7859 val_top1=57.99%  val_top5=90.11%  (2.2 min)
  ✓ New best: 57.99%  (saved)
  Epoch   8 [ 20/39]  loss=1.8811  top1=43.95%  lr=0.01000
  Epoch   8 [ 39/39]  loss=1.8937  top1=40.50%  lr=0.01000
Epoch   8/300  tr_loss=1.8937 tr_top1=40.50%  val_loss=1.8728 val_top1=53.73%  val_top5=87.60%  (3.3 min)
  Epoch   9 [ 20/39]  loss=2.1113  top1=36.17%  lr=0.01000
  Epoch   9 [ 39/39]  loss=1.9649  top1=37.74%  lr=0.01000
Epoch   9/300  tr_loss=1.9649 tr_top1=37.74%  val_loss=1.8135

In [ ]:
# ── 9. Training curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_top1'], label='Train Top-1')
axes[0].plot(history['val_top1'],   label='Val Top-1')
axes[0].plot(history['val_top5'],   label='Val Top-5', linestyle='--')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].axvline(best_epoch - 1, color='red', linestyle=':', label='Best epoch')

axes[1].plot(history['train_loss'], label='Train Loss')
axes[1].plot(history['val_loss'],   label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'RTD-Net on AID 50/50 — Best Val Top-1: {best_top1:.2f}%', fontsize=13)
plt.tight_layout()
plt.savefig(str(save_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved training curves.')

In [ ]:
# ── 10. Final evaluation with best checkpoint ─────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}  (val_top1={ckpt["val_top1"]:.2f}%)')

# Gather predictions
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(imgs)
        y_true.extend(labels.numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())

print('\n' + classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Normalised Confusion Matrix — RTD-Net on AID 50/50')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(str(save_dir / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('All done!')